download from companies marketcap

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import os
from datetime import date

# --- Configuração ---
URL = "https://companiesmarketcap.com/largest-companies-by-revenue/?download=csv"
OUTPUT_DIR = "../data/raw/"
FILE_PREFIX = "top_companies_revenue"
# --------------------

today = date.today().strftime("%Y%m%d")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}_{today}.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

response = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()

with open(OUTPUT_PATH, "wb") as f:
    f.write(response.content)

print(f"Arquivo salvo em: {OUTPUT_PATH}")
print(f"Tamanho: {len(response.content):,} bytes")


Arquivo salvo em: ../data/raw/top_companies_revenue_20260816.csv
Tamanho: 767,437 bytes


create interim dataset with standarlized columns

In [3]:
import glob
files = sorted(glob.glob("../data/raw/top_companies_revenue_*.csv"))
raw_data_file = files[-1]

print(f"Arquivo mais recente encontrado: {raw_data_file}")

import pandas as pd
df = pd.read_csv(raw_data_file)

std_columns = {
    "Rank": "rank",
    "Name": "company_name",
    "Symbol": "stock_symbol",
    "revenue_ttm": "revenue_ttm",
    "price (USD)": "price_usd",
    "country": "company_country",
}
df = df.rename(columns=std_columns)

replacement_char = {
    ",": ". ",
    ";": ". ",
    ":": ". ",
    "_": "-",
    "[": "(",
    "]": ")",
    "{": "(",
    "}": ")",
    "\"": "",
    "%": " percent ",
    "$": " dollar ",
    "#": " number ",
    "@": " at ",
    "!": "",
    "*": "",
    "+": " plus ",
    "=": " equals ",
    "<": " less than ",
    ">": " greater than ",
    "/": " slash ",
    "\\": " backslash ",
    "|": " pipe ",
    "~": " tilde "
}

for char in replacement_char.keys():
    df["company_name"] = df["company_name"].str.replace(char, replacement_char[char], regex=False)
    df["stock_symbol"] = df["stock_symbol"].str.replace(char, replacement_char[char], regex=False)

df["company_name"] = df["company_name"].str.lower()
display(df)


Arquivo mais recente encontrado: ../data/raw\top_companies_revenue_20260816.csv


,rank,company_name,stock_symbol,revenue_ttm,price_usd,company_country
0,1,amazon,AMZN,742776000000,262.65000,United States
1,2,walmart,WMT,725305000000,115.27000,United States
2,3,saudi aramco,2222.SR,459047318045,7.12497,Saudi Arabia
3,4,apple,AAPL,451442000000,305.93000,United States
4,5,unitedhealth,UNH,449713000000,401.73000,United States
...,...,...,...,...,...,...
11256,11257,kinnevik,KINV-A.ST,-879716521,6.83646,Sweden
11257,11258,burford capital,BUR,-1543511000,4.37000,Guernsey
11258,11259,banque nationale de belgique,BNB.BR,-1768524292,438.40400,Belgium
11259,11260,secure energy services,SES.TO,-2151095178,17.50330,Canada


Get GICS-like classification from FinanceDatabase

In [85]:
import pandas as pd
import financedatabase as fd

equities = fd.Equities()
p = equities.select(
    #only_primary_listing=True
)
p = p.reset_index()

# save P as csv
p.to_csv("./data/raw/equities.csv", index=False)



In [86]:
fd_dataset = pd.read_csv("./data/raw/equities.csv")

# merge the two datasets on stock_symbol, input the following columns from fd_dataset: sector, industry, sub_industry, website, zipcode

companies = pd.merge(
    df,
    fd_dataset[["symbol", "sector", "industry_group", "industry", "website", "zipcode"]],
    how="left",
    left_on="stock_symbol",
    right_on="symbol",
)

companies = pd.merge(df, fd_dataset[["symbol", "sector", "industry_group", "industry", "website", "zipcode"]], how="left", left_on="stock_symbol", right_on="symbol")

# delete the symbol column from companies
companies = companies.drop(columns=["symbol", "gics_sector", "gics_industry_group", "gics_industry"])

In [88]:
display(companies)
companies.to_csv(r"..\notebooks\data\raw\01_data_after_first_financialdata.csv", index=False)

,rank,company_name,stock_symbol,revenue_ttm,price_usd,company_country,sector,industry_group,industry,website,zipcode
0,1,amazon,AMZN,742776000000,262.65000,United States,Consumer Discretionary,Retailing,Internet & Direct Marketing Retail,http://www.amazon.com,98109-5210
1,2,walmart,WMT,725305000000,115.27000,United States,Consumer Staples,Household & Personal Products,Household Products,http://www.stock.walmart.com,72716
2,3,saudi aramco,2222.SR,459047318045,7.12497,Saudi Arabia,Energy,Energy,"Oil, Gas & Consumable Fuels",http://www.aramco.com,31311
3,4,apple,AAPL,451442000000,305.93000,United States,Information Technology,Technology Hardware & Equipment,"Electronic Equipment, Instruments & Components",http://www.apple.com,95014
4,5,unitedhealth,UNH,449713000000,401.73000,United States,Health Care,Health Care Equipment & Services,Health Care Providers & Services,http://www.unitedhealthgroup.com,55343
...,...,...,...,...,...,...,...,...,...,...,...
11256,11257,kinnevik,KINV-A.ST,-879716521,6.83646,Sweden,Financials,Diversified Financials,Capital Markets,http://www.kinnevik.com,103 13
11257,11258,burford capital,BUR,-1543511000,4.37000,Guernsey,Financials,Diversified Financials,Capital Markets,http://www.burfordcapital.com,GY1 2NP
11258,11259,banque nationale de belgique,BNB.BR,-1768524292,438.40400,Belgium,Financials,Banks,Banks,http://www.nbb.be,1000
11259,11260,secure energy services,SES.TO,-2151095178,17.50330,Canada,Energy,Energy,Energy Equipment & Services,http://www.secure-energy.com,T2P 1N2
